This notebook applies the finite state machine to the daily multimodal dataset for each participant and records the resulting daily behavioral states and transitions

Imports and initial configuration

In [ ]:
import pandas as pd
import numpy as np

# Load shared project paths and FSM configuration
%run 00_config.ipynb


# =========================
# LOAD DAILY DATASET
# =========================

DAILY_DATASET_FILE = OUTPUT_DIR / "daily_dataset.xlsx"

print("Daily dataset found:", DAILY_DATASET_FILE.exists())

daily_dataset = pd.read_excel(
    DAILY_DATASET_FILE
)

daily_dataset["date"] = pd.to_datetime(
    daily_dataset["date"]
)


# =========================
# INITIALIZE SIMULATION LOG
# =========================

all_users_simulation_log = []

print("Daily dataset shape:", daily_dataset.shape)

Simulation: The FSM is applied separately to each participant. Participant-specific temporal configurations and profile-specific thresholds are used to determine the initial state and subsequent daily state transitions.

In [ ]:
# Get all participant IDs
all_user_ids = daily_dataset["user_id"].unique()

for USER_ID in all_user_ids:

    # Select and chronologically sort the participant's data
    user_data = daily_dataset[
        daily_dataset["user_id"] == USER_ID
    ].copy()

    user_data = user_data.sort_values(
        "date"
    ).reset_index(drop=True)

    # Number of available days
    n_days = user_data["date"].nunique()

    user_config = choose_user_config(n_days)

    # Skip participants without enough data
    if user_config is None:
        print(
            f"Participant {USER_ID}: skipped "
            f"(insufficient days: {n_days})"
        )
        continue

    # Participant BMI profile
    user_profile = user_data["bmi_profile"].iloc[0]

    thresholds = get_profile_thresholds(user_profile)

    # Temporal configuration
    onboarding_days = user_config["onboarding_days"]

    max_transition_days = user_config["max_transition_days"]

    # Keep a 7-day window because MVPA thresholds are weekly
    window_days = 7

    # Separate onboarding and simulation periods
    onboarding_data = user_data.iloc[:onboarding_days].copy()

    simulation_data = user_data.iloc[onboarding_days:].copy()

    if len(simulation_data) == 0:
        continue

    # ==================================================
    # ONBOARDING
    # ==================================================

    baseline_hr = onboarding_data["hr_median"].median()

    baseline_rmssd = onboarding_data["rmssd_median"].median()

    baseline_stress = onboarding_data["avg_stress"].mean()

    hr_mad = np.median(
        np.abs(
            onboarding_data["hr_median"] - baseline_hr
        )
    )

    rmssd_mad = np.median(
        np.abs(
            onboarding_data["rmssd_median"] - baseline_rmssd
        )
    )

    mean_steps_onboarding = onboarding_data["steps"].mean()

    mvpa_onboarding = onboarding_data["mvpa_minutes"].sum()

    if mean_steps_onboarding > 0:

        cv_steps_onboarding = (
            onboarding_data["steps"].std()
            / mean_steps_onboarding
        ) * 100

    else:
        cv_steps_onboarding = 0.0

    # ==================================================
    # INITIAL STATE
    # ==================================================

    if mean_steps_onboarding >= thresholds["steps_transition"]:

        if cv_steps_onboarding <= thresholds["cv_regular"]:
            current_state = "Regularly Active"

        else:
            current_state = "Irregularly Active"

    else:

        if baseline_stress > 5:
            current_state = "Sedentary Under Pressure"

        else:
            current_state = "Stable Sedentary"

    previous_stable_state = current_state

    day_transitioning = 0

    # ==================================================
    # INITIAL LOG
    # ==================================================

    all_users_simulation_log.append({

        "user_id": USER_ID,

        "date": onboarding_data["date"].iloc[-1].strftime("%Y-%m-%d"),

        "previous_state": "None (Onboarding Init)",

        "new_state": current_state,

        "reason": (
            f"Initial state set via onboarding "
            f"steps ({mean_steps_onboarding:.0f}) "
            f"and stress ({baseline_stress:.2f})."
        ),

        "window_complete": len(onboarding_data) >= window_days,

        "day_in_transition": day_transitioning,

        "steps_today": onboarding_data["steps"].iloc[-1],

        "mvpa_today": onboarding_data["mvpa_minutes"].iloc[-1],

        "mvpa_last7days": mvpa_onboarding,

        "mean_steps_last7days": mean_steps_onboarding,

        "cv_last7days_pct": round(cv_steps_onboarding, 2),

        "stress_today": round(
            onboarding_data["avg_stress"].iloc[-1],
            2
        ),

        "hr_today": onboarding_data["hr_median"].iloc[-1],

        "rmssd_today": onboarding_data["rmssd_median"].iloc[-1],

        "stress_high_trigger": False,

        "activity_daily_trigger": False
    })

    # ==================================================
    # DAILY SIMULATION
    # ==================================================

    history = onboarding_data.copy()

    for _, current_day in simulation_data.iterrows():

        # Add the current day to the participant history
        history = pd.concat(
            [history, current_day.to_frame().T],
            ignore_index=True
        )

        # Weekly window
        last_7_days = history.tail(window_days)

        # Check whether a complete 7-day window is available
        window_complete = len(last_7_days) >= window_days

        mvpa_last7days = last_7_days[
            "mvpa_minutes"
        ].sum()

        mean_steps_last7days = last_7_days[
            "steps"
        ].mean()

        steps_std = last_7_days["steps"].std()

        steps_mean = last_7_days["steps"].mean()

        if pd.isna(steps_std) or steps_mean == 0:
            cv_steps_window = 0.0

        else:
            cv_steps_window = (
                steps_std / steps_mean
            ) * 100

        # ==================================================
        # CURRENT-DAY DATA
        # ==================================================

        stress_today = current_day["avg_stress"]

        hr_today = current_day["hr_median"]

        rmssd_today = current_day["rmssd_median"]

        # ==================================================
        # TRIGGERS
        # ==================================================

        stress_high = (
            stress_today > 5
        ) and (

            (
                rmssd_today < (
                    baseline_rmssd
                    - thresholds["mad_k"] * rmssd_mad
                )
            )

            or

            (
                hr_today > (
                    baseline_hr
                    + thresholds["mad_k"] * hr_mad
                )
            )
        )

        activity_trigger = (

            current_day["steps"]
            >= thresholds["steps_transition"]

            or

            current_day["mvpa_minutes"]
            >= thresholds["mvpa_transition"]
        )

        new_state = current_state

        if current_state in ["Regularly Active", "Irregularly Active"]:
            reason = "No changes observed in activity or regularity levels."
        else:
            reason = "No changes observed in stress or activity levels."

        # ==================================================
        # FSM RULES
        # ==================================================

        # Stable Sedentary
        if current_state == "Stable Sedentary":

            if activity_trigger:

                new_state = "Stable Sedentary Transitioning"

                reason = (
                    f"Activity trigger "
                    f"(Steps: {current_day['steps']} "
                    f"OR MVPA: {current_day['mvpa_minutes']})"
                )

            elif stress_high:

                new_state = "Sedentary Under Pressure"

                reason = (
                    f"Stress trigger "
                    f"(Stress: {stress_today:.2f})"
                )

        # Sedentary Under Pressure
        elif current_state == "Sedentary Under Pressure":

            if activity_trigger:

                new_state = (
                    "Sedentary Under Pressure Transitioning"
                )

                reason = (
                    f"Activity trigger under pressure "
                    f"(Steps: {current_day['steps']} "
                    f"OR MVPA: {current_day['mvpa_minutes']})"
                )

            elif not stress_high:

                new_state = "Stable Sedentary"

                reason = (
                    "Stress normalized and autonomic "
                    "balance restored."
                )

        # Transitioning states
        elif "Transitioning" in current_state:

            day_transitioning += 1

            # Active consolidation requires a complete 7-day window
            if (
                window_complete
                and
                mvpa_last7days >= thresholds["mvpa_active"]
            ):

                if cv_steps_window <= thresholds["cv_regular"]:

                    new_state = "Regularly Active"

                    reason = (
                        f"Progressed to Active: "
                        f"MVPA 7d ({mvpa_last7days} min) "
                        f"and Regular "
                        f"(CV: {cv_steps_window:.1f}%)"
                    )

                else:

                    new_state = "Irregularly Active"

                    reason = (
                        f"Progressed to Active: "
                        f"MVPA 7d ({mvpa_last7days} min) "
                        f"but Irregular "
                        f"(CV: {cv_steps_window:.1f}%)"
                    )

            elif day_transitioning >= max_transition_days:

                if "Sedentary" in previous_stable_state:

                    if stress_high:
                        new_state = "Sedentary Under Pressure"

                    else:
                        new_state = "Stable Sedentary"

                else:

                    if cv_steps_window <= thresholds["cv_regular"]:
                        new_state = "Regularly Active"

                    else:
                        new_state = "Irregularly Active"

                reason = (
                    f"Transition timeout "
                    f"({max_transition_days} days)."
                )

            elif (
                window_complete
                and
                mvpa_last7days < thresholds["mvpa_low"]
            ):

                if stress_high:
                    new_state = "Sedentary Under Pressure"

                else:
                    new_state = "Stable Sedentary"

                reason = (
                    f"Activity dropped heavily: "
                    f"MVPA 7d ({mvpa_last7days} min)"
                )

            else:

                reason = (
                    f"Maintained Transition: "
                    f"MVPA 7d ({mvpa_last7days} min). "
                    f"Day {day_transitioning}/"
                    f"{max_transition_days}."
                )

        # Regularly Active
        elif current_state == "Regularly Active":

            if (
                window_complete
                and
                mvpa_last7days < thresholds["mvpa_active"]
            ):

                new_state = "Regularly Active Transitioning"

                reason = (
                    f"Activity drop: "
                    f"MVPA 7d ({mvpa_last7days} min)"
                )

            elif cv_steps_window > thresholds["cv_regular"]:

                new_state = "Irregularly Active"

                reason = (
                    f"Regularity lost: "
                    f"CV ({cv_steps_window:.1f}%)"
                )

        # Irregularly Active
        elif current_state == "Irregularly Active":

            if (
                window_complete
                and
                mvpa_last7days < thresholds["mvpa_active"]
            ):

                new_state = "Irregularly Active Transitioning"

                reason = (
                    f"Activity drop: "
                    f"MVPA 7d ({mvpa_last7days} min)"
                )

            elif cv_steps_window <= thresholds["cv_regular"]:

                new_state = "Regularly Active"

                reason = (
                    f"Regularity gained: "
                    f"CV ({cv_steps_window:.1f}%)"
                )

        # ==================================================
        # UPDATE STATE MEMORY
        # ==================================================

        if (
            "Transitioning" in new_state
            and
            "Transitioning" not in current_state
        ):

            previous_stable_state = current_state

            day_transitioning = 1

        if "Transitioning" not in new_state:
            day_transitioning = 0

        # ==================================================
        # SAVE DAILY LOG
        # ==================================================

        all_users_simulation_log.append({

            "user_id": USER_ID,

            "date": current_day["date"].strftime("%Y-%m-%d"),

            "previous_state": current_state,

            "new_state": new_state,

            "reason": reason,

            "window_complete": window_complete,

            "day_in_transition": day_transitioning,

            "steps_today": current_day["steps"],

            "mvpa_today": current_day["mvpa_minutes"],

            "mvpa_last7days": mvpa_last7days,

            "mean_steps_last7days": mean_steps_last7days,

            "cv_last7days_pct": round(
                cv_steps_window,
                2
            ),

            "stress_today": round(stress_today, 2),

            "hr_today": hr_today,

            "rmssd_today": rmssd_today,

            "stress_high_trigger": stress_high,

            "activity_daily_trigger": activity_trigger
        })

        current_state = new_state

print(
    f"Simulation completed. "
    f"{len(all_user_ids)} participants evaluated."
)

Compile and save simulation results (save simulation log)

In [ ]:
# Convert the accumulated simulation log into a DataFrame
simulation_results = pd.DataFrame(
    all_users_simulation_log
)

# Save simulation results
output_file = OUTPUT_DIR / "fsm_simulation_all_users.xlsx"

simulation_results.to_excel(
    output_file,
    index=False
)

print("Simulation results saved successfully.")
print(f"Saved to: {output_file}")
print(f"Total simulation records: {len(simulation_results)}")

simulation_results.head(20)